In [0]:
select * from workspace.default.sales_data limit 100;

--QUESTION 1
-- 1.Return every column for all transactions in the Electronics category.
SELECT*
FROM workspace.default.sales_data
WHERE `Product Category` = 'Electronics';

--2.Return Customer ID, Age and Total Amount for all female customers, sorted by Total Amount from highest to lowest.
 SELECT `Customer ID`, `Age`, `Total Amount`
 FROM  workspace.default.sales_data
 WHERE `Gender` = 'Female'
 ORDER BY `Total Amount` DESC; 

--3.Show the 10 highest-value transactions.
SELECT*
FROM workspace.default.sales_data
ORDER BY `Total Amount` DESC
LIMIT 10;

--4. List all transactions where Quantity is 4 and Price per Unit is 500.
SELECT*
FROM workspace.default.sales_data
WHERE `Quantity` = '4' AND `Price per Unit` = '500';

--5. Return the distinct product categories in the dataset.
SELECT DISTINCT `Product Category`
FROM  workspace.default.sales_data;

--QUESTION 2
--6. How many transactions are there in total?
SELECT COUNT(*) AS transaction_count
FROM workspace.default.sales_data;

--7.  What is the total revenue across all transactions?
SELECT SUM(`Total Amount`) AS total_revenue
FROM workspace.default.sales_data;

--8.  What is the average customer age, rounded to 1 decimal place?
SELECT ROUND(AVG(Age), 1) AS avg_age
FROM workspace.default.sales_data;

--9. What is the average spend per transaction, rounded to 2 decimals?
SELECT ROUND(AVG(`Total Amount`), 2) AS avg_spend 
FROM workspace.default.sales_data;

--10.Return the minimum and maximum Total Amount in a single query.
SELECT MIN(`Total Amount`) AS min_amount,
       MAX(`Total Amount`) AS max_amount
FROM workspace.default.sales_data;

--QUESTION 3
--11. Total revenue per product category, sorted highest to lowest
SELECT SUM(`Total Amount`) AS product_revenue
FROM workspace.default.sales_data
GROUP BY `Product Category`
ORDER BY product_revenue DESC;

--12. For each gender, return the transaction count and the average spend
SELECT `Gender`, COUNT(*) AS transaction_count , AVG(`Total Amount`) AS avg_spend 
FROM workspace.default.sales_data
GROUP BY `Gender`;

--13. Average customer age per product category
SELECT AVG(`Age`) AS avg_age, `Product Category`
FROM workspace.default.sales_data
GROUP BY `Product Category`;

--14. Total quantity sold per product category.
SELECT SUM(`Quantity`) AS quantity_revenue, `Product Category`
FROM  workspace.default.sales_data
GROUP BY `Product Category`;
 
 --15. Which product categories have total revenue above 150,000? (USE HAVING)
 SELECT SUM(`Total Amount`) AS product_revenue, `Product Category`
 FROM workspace.default.sales_data
 GROUP BY `Product Category`
 HAVING SUM(`Total Amount`) > 150000
 ORDER BY product_revenue DESC;

 --16. Count the number of transactions per calendar month across the dataset.
 SELECT date_format(Date, 'yyyy-MM') AS year_month,
        COUNT(*) AS transaction_count
 FROM workspace.default.sales_data
 GROUP BY date_format(Date, 'yyyy-MM')
 ORDER BY year_month;

--QUESTION 4
--17. Bucket customers into age groups (18-29, 30-44, 45+) and count how many transactions fall in each.
SELECT CASE
           WHEN Age < 30 THEN '18-29'
           WHEN Age < 45 THEN '30-44'
           ELSE '45+'
           END AS age_group,
           COUNT(*) AS transaction_count
FROM workspace.default.sales_data
GROUP BY CASE
             WHEN Age < 30 THEN '18-29'
             WHEN Age < 45 THEN '30-44'
             ELSE '45+'
             END
ORDER BY age_group; 

--18.Label each transaction as High (>= 500), Medium (100-499), or Low (< 100) by Total Amount, and count how many fall in each band.
SELECT CASE
         WHEN `Total Amount` >= 500 THEN 'High'
         WHEN `Total Amount` BETWEEN 100 AND 499 THEN 'Medium'
         ELSE 'Low'
         END AS transaction_count
FROM workspace.default.sales_data
GROUP BY CASE
          WHEN `Total Amount` >= 500 THEN 'High'
           WHEN `Total Amount` BETWEEN 100 AND 499 THEN 'Medium'
           ELSE 'Low'
           END 
ORDER BY transaction_count DESC;

--19. Total revenue by quarter for the year 2023.
SELECT quarter(`Date`) AS quarter,
     SUM(`Total Amount`)
FROM workspace.default.sales_data

--20. Compare the average spend of customers over 40 versus 40 and under.
SELECT CASE 
          WHEN `Age` > 40 THEN 'Over 40'
          ELSE '40 and under' 
          END AS age_segment,
          ROUND(AVG(`Total Amount`), 2) AS avg_spend,
          COUNT(*) AS transaction_count
FROM workspace.default.sales_data
GROUP BY CASE 
          WHEN `Age` > 40 THEN 'Over 40'
          ELSE '40 and under' 
          END;

--QUESTION 5
--21. Within each product category, rank transactions by Total Amount (highest = rank 1)
SELECT `Transaction ID`, 
       `Product Category`,
      `Total Amount`,
       RANK() OVER (PARTITION BY `Product Category` ORDER BY `Total Amount` DESC) AS category_rank
FROM workspace.default.sales_data;

--22. For each transaction, show its Total Amount, the average for its category, and the difference between them
SELECT `Transaction ID`,
       `Product Category`,
       `Total Amount`, 
       ROUND(AVG(`Total Amount`) OVER (PARTITION BY `Product Category`), 2) AS category_avg,
       ROUND(`Total Amount` - AVG(`Total Amount`) OVER (PARTITION BY `Product Category`), 2) AS diff_from_avg
FROM workspace.default.sales_data;

--23. Produce a running total of revenue ordered by Date .
SELECT Date,
`Total Amount`,
SUM(`Total Amount`) OVER (ORDER BY Date
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS 
running_revenue
FROM workspace.default.sales_data;

--24. Return the top 3 highest-value transactions per gender.
SELECT `Transaction ID`, Gender, `Total Amount`
FROM workspace.default.sales_data
QUALIFY ROW_NUMBER() OVER (PARTITION BY Gender
ORDER BY `Total Amount` DESC) <= 3;

--25.Show each product category's revenue and its percentage contribution to total rev
SELECT `Product Category`,
SUM(`Total Amount`) AS revenue,
ROUND(SUM(`Total Amount`) * 100.0 / SUM(SUM(`Total Amount`)) OVER (), 2) AS 
pct_of_total
FROM workspace.default.sales_data
GROUP BY `Product Category`
ORDER BY pct_of_total DESC;